# <font color="#418FDE" size="6.5" uppercase>**Advanced Neighbors**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Apply nearest-centroid classifiers and neighbor graph transformers. 
- Use Neighborhood Components Analysis to learn supervised feature transformations. 
- Evaluate limitations of neighbor methods in high-dimensional and connected workflows. 


## **1. Centroid Methods**

### **1.1. Nearest Centroid**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_01.jpg?v=1783965493" width="250">



>* Summarizes each class with a centroid
>* Classifies by the closest class profile

>* Scale features before comparing centroid distances
>* Centroids provide simple, explainable class profiles

>* Best for compact, well-separated classes
>* Struggles with complex or overlapping groups



In [ ]:
#@title Python Code - Nearest Centroid

# Demonstrate nearest centroid classification.
# Compare scaled and unscaled distances.
# Keep outputs short and visual.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny customer examples.
X_train = np.array([
    [25, 2, 4.8],
    [30, 3, 4.6],
    [80, 9, 3.2],
    [90, 8, 3.4],
])

# Store class labels clearly.
y_train = np.array([
    "casual",
    "casual",
    "frequent",
    "frequent",
])

# Define one new customer.
new_customer = np.array([
    [55, 4, 4.4],
])

# Validate expected feature counts.
assert X_train.shape[1] == new_customer.shape[1]

# Compute centroids for each class.
classes = np.unique(y_train)
centroids = np.vstack([
    X_train[y_train == label].mean(axis=0)
    for label in classes
])

# Predict using raw feature distances.
raw_distances = np.linalg.norm(
    centroids - new_customer,
    axis=1,
)
raw_prediction = classes[np.argmin(raw_distances)]

# Standardize features using training statistics.
means = X_train.mean(axis=0)
stds = X_train.std(axis=0)
stds = np.where(stds == 0, 1, stds)

# Transform training data and customer.
X_scaled = (X_train - means) / stds
new_scaled = (new_customer - means) / stds

# Recompute centroids after scaling.
scaled_centroids = np.vstack([
    X_scaled[y_train == label].mean(axis=0)
    for label in classes
])

# Predict using scaled feature distances.
scaled_distances = np.linalg.norm(
    scaled_centroids - new_scaled,
    axis=1,
)
scaled_prediction = classes[np.argmin(scaled_distances)]

# Print compact teaching results.
print("Nearest centroid without scaling:", raw_prediction)
print("Nearest centroid with scaling:", scaled_prediction)
print("Classes represented by centroids:", list(classes))

# Plot two features and centroids.
plt.figure(figsize=(6, 4))
for label in classes:
    mask = y_train == label
    plt.scatter(X_train[mask, 0], X_train[mask, 1], label=label)

# Add centroids and new customer.
plt.scatter(centroids[:, 0], centroids[:, 1], marker="X", s=160, label="centroids")
plt.scatter(new_customer[:, 0], new_customer[:, 1], marker="*", s=180, label="new customer")
plt.xlabel("Annual income, thousands of dollars")
plt.ylabel("Purchases last month")
plt.title("Nearest centroid uses class average profiles")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



### **1.2. Shrunken Centroids**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_02.jpg?v=1783965495" width="250">



>* Shrink centroids toward overall feature averages
>* Reduce noise and emphasize diagnostic signals

>* Shrink noisy features to improve generalization
>* Keep signals that consistently separate classes

>* Highlights key features for clearer decisions
>* Tune shrinkage to balance accuracy and simplicity



In [ ]:
#@title Python Code - Shrunken Centroids

# Demonstrate shrunken centroids without external machine learning libraries.
# Tiny synthetic data keeps the example fast.
# Shrinkage reduces noisy feature influence.

import numpy as np
import matplotlib.pyplot as plt

# Create reproducible two class data.
rng = np.random.default_rng(7)
n_per_class = 24
n_features = 12

# Only first features separate classes.
class_zero = rng.normal(0.0, 1.0, (n_per_class, n_features))
class_one = rng.normal(0.0, 1.0, (n_per_class, n_features))
class_one[:, :3] += np.array([2.2, 1.6, 1.1])

# Combine data and labels.
X = np.vstack([class_zero, class_one])
y = np.array([0] * n_per_class + [1] * n_per_class)
assert X.shape == (48, 12)

# Split deterministically into train and test.
indices = rng.permutation(len(y))
train_idx = indices[:36]
test_idx = indices[36:]

# Prepare train and test arrays.
X_train = X[train_idx]
y_train = y[train_idx]
X_test = X[test_idx]
y_test = y[test_idx]

# Compute ordinary class centroids.
classes = np.array([0, 1])
centroids = np.vstack([X_train[y_train == c].mean(axis=0) for c in classes])
overall = X_train.mean(axis=0)

# Shrink centroid differences toward overall mean.
shrink = 0.75
diff = centroids - overall
shrunk_diff = np.sign(diff) * np.maximum(np.abs(diff) - shrink, 0.0)
shrunk_centroids = overall + shrunk_diff

# Define a small nearest centroid predictor.
def predict_nearest_centroid(data, centers):
    distances = ((data[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    return classes[np.argmin(distances, axis=1)]

# Evaluate ordinary and shrunken centroids.
plain_pred = predict_nearest_centroid(X_test, centroids)
shrunk_pred = predict_nearest_centroid(X_test, shrunk_centroids)
plain_acc = np.mean(plain_pred == y_test)
shrunk_acc = np.mean(shrunk_pred == y_test)

# Count features still influencing each class.
active_features = np.sum(np.any(np.abs(shrunk_diff) > 0, axis=0))
print(f"Plain centroid accuracy: {plain_acc:.2f}")
print(f"Shrunken centroid accuracy: {shrunk_acc:.2f}")
print(f"Active features after shrinkage: {active_features} of {n_features}")

# Plot centroid differences before and after shrinkage.
feature_numbers = np.arange(1, n_features + 1)
plt.figure(figsize=(8, 4))
plt.plot(feature_numbers, diff[1], marker="o", label="ordinary class difference")
plt.plot(feature_numbers, shrunk_diff[1], marker="s", label="shrunken difference")
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("Feature number")
plt.ylabel("Class 1 centroid minus overall mean")
plt.title("Shrunken centroids suppress weak feature differences")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Centroid Limitations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_01_03.jpg?v=1783965488" width="250">



>* Centroids work best for compact, simple classes
>* Complex subgroups can make averages misleading

>* Feature scaling can distort centroid distances
>* Outliers and complex shapes limit centroids

>* High dimensions make centroid distances less reliable
>* Centroids miss local neighborhood structure



## **2. Neighbor Graphs**

### **2.1. K Neighbor Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_01.jpg?v=1783965477" width="250">



>* Connects each point to nearest similar points
>* Supports algorithms using explicit neighborhood structure

>* k balances local detail and graph stability
>* NCA improves class-based neighborhood structure

>* Scaling and dimensions shape neighbor reliability
>* Learned features make similarity task-relevant



In [ ]:
#@title Python Code - K Neighbor Graphs

# Build a tiny neighbor graph example.
# Compare raw and transformed feature spaces.
# Visualize fixed k nearest connections.

import numpy as np
import matplotlib.pyplot as plt

# Create two small labeled point groups.
rng = np.random.default_rng(7)
class_zero = rng.normal([0.0, 0.0], 0.35, (8, 2))
class_one = rng.normal([2.0, 0.7], 0.35, (8, 2))

# Stack points and class labels.
X_raw = np.vstack([class_zero, class_one])
y = np.array([0] * 8 + [1] * 8)
assert X_raw.shape == (16, 2)

# Apply a simple supervised-style transformation.
X_trans = X_raw @ np.array([[1.4, -0.2], [0.0, 0.5]])
assert X_trans.shape == X_raw.shape

# Define a small k value.
k = 3
n_points = X_raw.shape[0]
assert 0 < k < n_points

# Compute nearest neighbor indices manually.
def k_neighbors(X, k_value):
    distances = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
    np.fill_diagonal(distances, np.inf)
    return np.argsort(distances, axis=1)[:, :k_value]

# Build neighbor lists for both spaces.
raw_neighbors = k_neighbors(X_raw, k)
trans_neighbors = k_neighbors(X_trans, k)

# Count same-class neighbor connections.
def same_class_rate(neighbors, labels):
    matches = labels[neighbors] == labels[:, None]
    return matches.mean()

# Summarize graph quality compactly.
raw_rate = same_class_rate(raw_neighbors, y)
trans_rate = same_class_rate(trans_neighbors, y)
print(f"Points: {n_points}, k: {k}")
print(f"Raw same-class neighbor rate: {raw_rate:.2f}")
print(f"Transformed same-class neighbor rate: {trans_rate:.2f}")

# Draw the transformed k neighbor graph.
colors = np.where(y == 0, "tab:blue", "tab:orange")
plt.figure(figsize=(6, 4))
plt.scatter(X_trans[:, 0], X_trans[:, 1], c=colors, s=70)

# Add graph edges from each point.
for i in range(n_points):
    for j in trans_neighbors[i]:
        plt.plot([X_trans[i, 0], X_trans[j, 0]], [X_trans[i, 1], X_trans[j, 1]], color="gray", alpha=0.35)

# Label the teaching plot clearly.
plt.title("K Neighbor Graph After Feature Transformation")
plt.xlabel("Transformed feature 1")
plt.ylabel("Transformed feature 2")
plt.tight_layout()
plt.show()



### **2.2. Radius Neighbor Graphs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_02.jpg?v=1783965475" width="250">



>* Connect points within a chosen distance
>* Show density and learned class neighborhoods

>* Radius controls local graph connections
>* Match radius to scale, metric, transformation

>* Check class consistency after supervised transformations
>* Validate radius choices in high dimensions



In [ ]:
#@title Python Code - Radius Neighbor Graphs

# Radius graphs connect points within distance.
# Small examples reveal local density differences.
# This script avoids external machine learning libraries.

# Import numerical and plotting tools.
import numpy as np
import matplotlib.pyplot as plt

# Create two dense groups and one isolated point.
points = np.array([
    [0.0, 0.0], [0.4, 0.1],
    [0.2, 0.5], [3.0, 3.0],

    [3.3, 3.1], [3.1, 3.5],
    [6.0, 0.2]
])

# Store simple class labels for coloring.
labels = np.array([0, 0, 0, 1, 1, 1, 2])
radius = 0.75

# Validate the tiny coordinate table.
assert points.shape == (7, 2)
assert labels.shape[0] == points.shape[0]

# Compute all pairwise Euclidean distances.
diffs = points[:, None, :] - points[None, :, :]
distances = np.sqrt(np.sum(diffs * diffs, axis=2))

# Build a radius neighbor adjacency matrix.
adjacency = (distances <= radius) & (distances > 0)
neighbor_counts = adjacency.sum(axis=1)

# Summarize graph density without printing arrays.
edge_count = int(adjacency.sum() // 2)
isolated_count = int(np.sum(neighbor_counts == 0))

# Print concise teaching results.
print(f"Radius used: {radius} coordinate units")
print(f"Undirected edges found: {edge_count}")
print(f"Isolated observations: {isolated_count}")
print(f"Neighbor counts: {neighbor_counts.tolist()}")

# Draw points and radius-based connections.
fig, ax = plt.subplots(figsize=(6, 4))
colors = np.array(["tab:blue", "tab:orange", "tab:green"])

# Plot every radius edge once.
for i in range(points.shape[0]):
    for j in range(i + 1, points.shape[0]):
        if adjacency[i, j]:
            ax.plot(points[[i, j], 0], points[[i, j], 1], color="gray")

# Plot observations after drawing edges.
ax.scatter(points[:, 0], points[:, 1], c=colors[labels], s=90)
ax.set_title("Radius Neighbor Graph")
ax.set_xlabel("Feature 1")

# Finish the labeled coordinate plot.
ax.set_ylabel("Feature 2")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### **2.3. KNeighborsTransformer**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_02_03.jpg?v=1783965479" width="250">



>* Turns data into sparse neighbor graphs
>* Supports clustering, visualization, and semi-supervised learning

>* NCA learns class-separating feature transformations
>* KNeighborsTransformer graphs improved neighborhood relationships

>* Reusable graphs support efficient modular workflows
>* Graph quality depends on careful neighbor choices



## **3. Metric Learning Limits**

### **3.1. NCA Feature Transformation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_01.jpg?v=1783965481" width="250">



>* NCA learns better distances for classification
>* Bad labels can mislead learned transformations

>* NCA reshapes features to improve neighborhoods
>* Small or messy data can cause overfitting

>* NCA still relies on meaningful neighbors
>* Validate pipelines to avoid leakage and overfitting



### **3.2. NCA KNN Pipeline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_02.jpg?v=1783965483" width="250">



>* NCA learns label-guided features before KNN
>* KNN classifies using the improved distance space

>* Fit NCA only within training splits
>* Evaluate the full pipeline together

>* NCA cannot fix noisy high-dimensional neighborhoods
>* Pipelines add cost and remain limited



### **3.3. High Dimensional Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_B/image_03_03.jpg?v=1783965486" width="250">



>* High dimensions make nearness less meaningful
>* NCA cannot remove all noisy variation

>* Metric learning can overfit small datasets
>* Validate features to avoid fragile neighborhoods

>* High-dimensional neighbor graphs can be unstable
>* Evaluate stability, meaning, and simpler alternatives



# <font color="#418FDE" size="6.5" uppercase>**Advanced Neighbors**</font>


In this lecture, you learned to:
- Apply nearest-centroid classifiers and neighbor graph transformers. 
- Use Neighborhood Components Analysis to learn supervised feature transformations. 
- Evaluate limitations of neighbor methods in high-dimensional and connected workflows. 

In the next Module (Module 14), we will go over 'Gaussian Cross'